# 06 — Primary unsupervised model

This notebook fits the label-free detector on the **early calibration slice**
and learns candidate score thresholds from the disjoint **late calibration
slice**. It produces four complementary primary channels:

- rapid self evidence from empirical calibration tails;
- persistent drift evidence from a causal CUSUM;
- peer deviation within a valid Telecom peer group;
- common-mode evidence at physical topology scopes.

Robust PCA and dispersion change remain explicit challengers. Isolation
Forest is compared in three auditable forms: the original base features,
the enhanced temporal features, and self-plus-topology context. Every
partition is scored by the same shared function and all
resolved policy values are frozen beside the model. No fault or ticket label
is read here.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import joblib
import shutil
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from telco_anomaly.detectors import (
    calibration_thresholds,
    fit_contextual_isolation_forest,
    fit_residual_bundle,
    fit_topology_reference,
    merge_score_files,
    score_residual_file,
    score_partition_file,
    score_topology_file,
)
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    load_config,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
default_core_runs = {
    "synthetic_pon": "synthetic_pon_core_v2",
    "ran_pm": "ran_pm_v1",
    "microsoft_optical": "microsoft_optical_v1",
}
if DATASET not in default_core_runs:
    raise ValueError(f"Model fitting is not configured for {DATASET!r}")
legacy_core = os.getenv("PON_CORE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_features = os.getenv("PON_FEATURE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_models = os.getenv("PON_MODEL_RUN_ID") if DATASET == "synthetic_pon" else None
CORE_RUN_ID = os.getenv("TELCO_CORE_RUN_ID", legacy_core or default_core_runs[DATASET])
FEATURE_RUN_ID = os.getenv(
    "TELCO_FEATURE_RUN_ID", legacy_features or f"{DATASET}_features_v4"
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", legacy_models or f"{DATASET}_models_v5"
)

RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
FEATURE_ROOT = DATA_ROOT / "features" / DATASET / FEATURE_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "models" / DATASET / MODEL_RUN_ID

ALERT_POLICY = load_config("alert_policy", project_root=PROJECT_ROOT)
TOPOLOGY_POLICY = load_config("topology", project_root=PROJECT_ROOT)
FEATURE_CONFIG = load_config("features", project_root=PROJECT_ROOT)
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
core_manifest = read_json(CORE_ROOT / "manifest.json")
feature_manifest = read_json(FEATURE_ROOT / "feature_manifest.json")
CURRENT_FEATURES_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "features.py"
)
CURRENT_DETECTORS_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py"
)
CURRENT_FEATURE_CONFIG_SHA256 = file_sha256(
    PROJECT_ROOT / "configs" / "features.yml"
)
require_same(
    feature_manifest,
    core_fingerprint=core_manifest["fingerprint"],
    features_module_sha256=CURRENT_FEATURES_SHA256,
    feature_config_sha256=CURRENT_FEATURE_CONFIG_SHA256,
)
SCRATCH_PARENT = Path(os.getenv(
    "TELCO_WORK_ROOT",
    "/content" if "google.colab" in sys.modules else tempfile.gettempdir(),
))
SCRATCH_PARENT.mkdir(parents=True, exist_ok=True)
SCRATCH_FREE_GB = shutil.disk_usage(SCRATCH_PARENT).free / 1024**3
MINIMUM_SCRATCH_GB = float(os.getenv("TELCO_MIN_SCRATCH_GB", "4"))
if SCRATCH_FREE_GB < MINIMUM_SCRATCH_GB:
    raise OSError(
        f"Notebook 06 needs at least {MINIMUM_SCRATCH_GB:.0f} GB free in "
        f"{SCRATCH_PARENT}; only {SCRATCH_FREE_GB:.1f} GB is available. "
        "Set TELCO_WORK_ROOT to a larger local disk."
    )

if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("Model fitting must use the truth-unmounted run")

feature_paths = {
    name: FEATURE_ROOT / values["features"]
    for name, values in feature_manifest["partitions"].items()
}
display(pd.Series({
    "calibration_fit_features": str(feature_paths["calibration_fit"]),
    "calibration_threshold_features": str(feature_paths["calibration_threshold"]),
    "dataset": DATASET,
    "development_features": str(feature_paths["development"]),
    "model_output": str(OUTPUT_ROOT),
    "local_scratch": str(SCRATCH_PARENT),
    "local_scratch_free_gb": round(SCRATCH_FREE_GB, 1),
}, name="value").to_frame())


## 2. Fit the frozen self-history reference


In [ ]:
MAX_TRAINING_ROWS = int(os.getenv("PON_MAX_TRAINING_ROWS", "150000"))
ISOLATION_POLICY = FEATURE_CONFIG["isolation_forest"]
ISOLATION_TREES = int(os.getenv(
    "PON_ISOLATION_TREES", str(ISOLATION_POLICY["n_estimators"])
))
ISOLATION_MAX_SAMPLES = int(ISOLATION_POLICY["max_samples"])
ISOLATION_MAX_FEATURES = float(ISOLATION_POLICY["max_features"])
MODEL_SEED = int(ISOLATION_POLICY["random_seed"])
CADENCE_SECONDS = float(catalogue["expected_cadence_seconds"].dropna().mode().iloc[0])
REQUESTED_TOPOLOGY_FEATURES = TOPOLOGY_POLICY["group_policy"][
    "residual_features_by_dataset"
].get(DATASET, [])

FEATURE_MANIFEST_SHA256 = file_sha256(FEATURE_ROOT / "feature_manifest.json")
TOPOLOGY_CONFIG_SHA256 = file_sha256(PROJECT_ROOT / "configs" / "topology.yml")
expected_model_inputs = {
    "core_fingerprint": core_manifest["fingerprint"],
    "feature_manifest_sha256": FEATURE_MANIFEST_SHA256,
    "detectors_module_sha256": CURRENT_DETECTORS_SHA256,
    "topology_config_sha256": TOPOLOGY_CONFIG_SHA256,
}

if OUTPUT_ROOT.exists():
    model_manifest = read_json(OUTPUT_ROOT / "model_manifest.json")
    require_same(model_manifest, **expected_model_inputs)
    if file_sha256(OUTPUT_ROOT / "resolved_policy.json") != model_manifest["resolved_policy_sha256"]:
        raise ValueError("The frozen model policy no longer matches its manifest")
    bundle = joblib.load(OUTPUT_ROOT / "residual_bundle.joblib")
    print("Using existing immutable model:", OUTPUT_ROOT)
else:
    bundle = fit_residual_bundle(
        feature_paths["calibration_fit"],
        use_entity_reference=True,
        catalogue=catalogue,
        maximum_training_rows=MAX_TRAINING_ROWS,
        random_seed=MODEL_SEED,
        isolation_trees=ISOLATION_TREES,
        isolation_max_samples=ISOLATION_MAX_SAMPLES,
        isolation_max_features=ISOLATION_MAX_FEATURES,
        fit_multivariate=True,
    )

TOPOLOGY_FEATURES = [
    name for name in REQUESTED_TOPOLOGY_FEATURES
    if name in bundle["feature_columns"]
]

display(pd.Series({
    "training_rows": bundle["training_rows"],
    "health_features": len(bundle["feature_columns"]),
    "entity_reference": bundle["use_entity_reference"],
    "entity_reference_entities": (
        len(bundle["entity_centre"])
        if bundle["entity_centre"] is not None else 0
    ),
    "minimum_entity_reference_rows": (
        int(bundle["entity_reference_counts"].min().min())
        if bundle.get("entity_reference_counts") is not None else 0
    ),
    "empirical_tail_calibration": bool(bundle["tail_reference"]),
    "base_isolation_features": len(bundle["isolation_base_features"]),
    "temporal_isolation_features": len(bundle["feature_columns"]),
    "isolation_forest_fitted": bundle["isolation_forest_temporal"] is not None,
    "topology_residual_features": len(TOPOLOGY_FEATURES),
}, name="value").to_frame())


## 3. Define the single partition-scoring path


In [ ]:
def choose_peer_level(topology):
    policy = TOPOLOGY_POLICY["peer_policy"]
    minimum_group_size = int(policy["minimum_valid_peers"]) + 1
    minimum_coverage = float(policy["minimum_entity_coverage"])
    total_entities = topology["entity_id"].nunique()
    hierarchy = topology.groupby("group_type")["hierarchy_level"].median()
    configured = TOPOLOGY_POLICY["peer_policy"]["preferred_levels"]
    actual = topology["group_type"].drop_duplicates().tolist()
    candidates = [name for name in configured if name in actual]
    candidates += sorted(
        set(actual) - set(candidates),
        key=lambda name: hierarchy.get(name, -1), reverse=True,
    )
    for group_type in candidates:
        level = topology.loc[topology["group_type"].eq(group_type)]
        sizes = level.groupby("group_id")["entity_id"].nunique()
        eligible_groups = set(sizes.loc[sizes.ge(minimum_group_size)].index)
        covered = level.loc[level["group_id"].isin(eligible_groups), "entity_id"].nunique()
        if total_entities and covered / total_entities >= minimum_coverage:
            return group_type
    raise ValueError(
        "No topology level gives enough peers to the required share of entities"
    )


## 4. Fit topology evidence, score the required partitions and calibrate thresholds

Calibration-fit scores are computed once and reused to fit topology context.
Only a compact set of interpretable residuals is written for cross-entity
scoring. Temporary residual, topology and merge files are deleted as soon as
their consumer finishes. Only late-calibration and development scores are
published because downstream notebooks do not use training-slice scores.


In [ ]:
if not OUTPUT_ROOT.exists():
    with immutable_output_directory(OUTPUT_ROOT) as output:
        topology_path = CORE_ROOT / "topology_memberships.parquet"
        topology_enabled = topology_path.exists()
        topology_reason = (
            "available" if topology_enabled else "topology file unavailable"
        )
        topology = pd.read_parquet(topology_path) if topology_enabled else None
        if topology_enabled and len(TOPOLOGY_FEATURES) < 2:
            topology_enabled = False
            topology_reason = (
                "no usable dataset-specific topology feature registry"
            )
        topology_reference = pd.DataFrame()
        peer_level = None
        group_levels = []
        contextual_bundle = None
        contextual_status = "topology unavailable"

        # Fit the topology reference on the early calibration slice only.
        with tempfile.TemporaryDirectory(
            dir=SCRATCH_PARENT, prefix="telco-model-fit-"
        ) as fit_name:
            fit_workspace = Path(fit_name)
            fit_self = fit_workspace / "calibration_fit_self.parquet"
            fit_residuals = fit_workspace / "calibration_fit_residuals.parquet"
            score_residual_file(
                bundle,
                feature_paths["calibration_fit"],
                fit_self,
                cadence_seconds=CADENCE_SECONDS,
                dispersion_window_seconds=feature_manifest["dispersion_window_seconds"],
                cusum_allowance=ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"],
                residual_destination=fit_residuals if topology_enabled else None,
                residual_features=TOPOLOGY_FEATURES,
            )
            if topology_enabled:
                physical = topology.loc[
                    topology["group_family"].eq("physical_topology")
                ]
                try:
                    peer_level = choose_peer_level(physical)
                    group_levels = physical.sort_values(
                        "hierarchy_level", ascending=False
                    )["group_type"].drop_duplicates().tolist()
                    topology_reference = fit_topology_reference(
                        fit_residuals,
                        topology,
                        TOPOLOGY_FEATURES,
                        peer_group_type=peer_level,
                        group_types=group_levels,
                        min_peers=TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
                        min_group_entities=TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
                        min_group_fraction=TOPOLOGY_POLICY["group_policy"]["minimum_available_fraction"],
                    )
                    # Export only residuals that survived reference fitting.
                    TOPOLOGY_FEATURES = sorted(
                        topology_reference["leading_feature"].astype(str).unique()
                    )
                except ValueError as error:
                    topology_enabled = False
                    topology_reason = str(error)
                    topology_reference = pd.DataFrame()
                    peer_level = None
                    group_levels = []

            fit_policy = {
                "cusum_allowance": ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"],
                "topology_enabled": topology_enabled,
                "peer_group_type": peer_level,
                "group_types": group_levels,
                "min_peers": TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
                "min_group_entities": TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
                "min_group_fraction": TOPOLOGY_POLICY["group_policy"]["minimum_available_fraction"],
                "topology_features": TOPOLOGY_FEATURES,
            }
            if topology_enabled:
                fit_topology = fit_workspace / "calibration_fit_topology.parquet"
                fit_combined = fit_workspace / "calibration_fit_combined.parquet"
                score_topology_file(
                    fit_residuals, topology, topology_reference, fit_topology,
                    peer_group_type=peer_level,
                    group_types=group_levels,
                    min_peers=fit_policy["min_peers"],
                    min_group_entities=fit_policy["min_group_entities"],
                    min_group_fraction=fit_policy["min_group_fraction"],
                )
                fit_residuals.unlink()
                merge_score_files(fit_self, fit_topology, fit_combined)
                fit_self.unlink()
                fit_topology.unlink()
                try:
                    contextual_bundle = fit_contextual_isolation_forest(
                        fit_combined,
                        ISOLATION_POLICY["contextual_inputs"],
                        maximum_training_rows=MAX_TRAINING_ROWS,
                        random_seed=MODEL_SEED,
                        trees=ISOLATION_TREES,
                        maximum_samples=ISOLATION_MAX_SAMPLES,
                        maximum_features=ISOLATION_MAX_FEATURES,
                    )
                    contextual_status = "available"
                except ValueError as error:
                    contextual_status = f"unavailable: {error}"

        resolved_policy = {
            "alert_policy": ALERT_POLICY,
            "topology_policy": TOPOLOGY_POLICY,
            "cusum_allowance": ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"],
            "topology_enabled": topology_enabled,
            "peer_group_type": peer_level,
            "group_types": group_levels,
            "min_peers": TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
            "min_group_entities": TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
            "min_group_fraction": TOPOLOGY_POLICY["group_policy"]["minimum_available_fraction"],
            "topology_features": TOPOLOGY_FEATURES,
            "contextual_isolation_status": contextual_status,
        }

        combined_paths = {}
        local_scores = {}
        with tempfile.TemporaryDirectory(
            dir=SCRATCH_PARENT, prefix="telco-model-score-"
        ) as score_name:
            score_workspace = Path(score_name)
            for partition in ("calibration_threshold", "development"):
                print(f"Scoring {partition}")
                local_score = score_workspace / f"{partition}_scores.parquet"
                score_partition_file(
                    bundle,
                    feature_paths[partition],
                    local_score,
                    score_workspace / f"work_{partition}",
                    cadence_seconds=CADENCE_SECONDS,
                    dispersion_window_seconds=feature_manifest["dispersion_window_seconds"],
                    resolved_policy=resolved_policy,
                    topology=topology,
                    topology_reference=topology_reference,
                    contextual_isolation_bundle=contextual_bundle,
                )
                published = output / local_score.name
                shutil.copy2(local_score, published)
                combined_paths[partition] = published
                local_scores[partition] = local_score
                if partition == "development":
                    local_score.unlink()

            score_columns = set(
                pq.ParquetFile(local_scores["calibration_threshold"])
                .schema_arrow.names
            )
            channels = [
                name for name in (
                    "rapid_residual", "drift_cusum", "peer_deviation",
                    "group_common_mode", "dispersion_change", "pca_spe",
                    "isolation_forest_base",
                    "isolation_forest_temporal",
                    "isolation_forest_contextual",
                )
                if name in score_columns
            ]
            thresholds = calibration_thresholds(
                local_scores["calibration_threshold"],
                ALERT_POLICY["thresholds"]["candidate_quantiles"],
                block_column="entity_id",
                block_duration_seconds=ALERT_POLICY["thresholds"]["block_seconds"],
                minimum_block_rows=max(4, round(0.5 * 86_400 / CADENCE_SECONDS)),
                model_ids=channels,
            )
            local_scores["calibration_threshold"].unlink()

        bundle["contextual_isolation_bundle"] = contextual_bundle
        joblib.dump(bundle, output / "residual_bundle.joblib")
        bundle["feature_audit"].to_parquet(
            output / "feature_retention.parquet", index=False
        )
        if contextual_bundle is not None:
            contextual_bundle["feature_audit"].to_parquet(
                output / "contextual_feature_retention.parquet", index=False
            )
        thresholds.to_parquet(output / "calibration_thresholds.parquet", index=False)
        if len(topology_reference):
            topology_reference.to_parquet(
                output / "topology_reference.parquet", index=False
            )
        write_json(output / "resolved_policy.json", resolved_policy)
        model_manifest = {
            "dataset": DATASET,
            **expected_model_inputs,
            "resolved_policy_sha256": file_sha256(output / "resolved_policy.json"),
            "calibration_only_fit": True,
            "calibration_fit_fraction": feature_manifest["calibration_fit_fraction"],
            "calibration_threshold_start": feature_manifest["calibration_threshold_start"],
            "truth_files_read": [],
            "channels": channels,
            "primary_channels": [
                "rapid_residual", "drift_cusum", "peer_deviation", "group_common_mode"
            ],
            "challenger_channels": [
                "dispersion_change", "pca_spe",
                "isolation_forest_base", "isolation_forest_temporal",
                "isolation_forest_contextual",
            ],
            "score_files": {
                name: f"{name}_scores.parquet" for name in combined_paths
            },
            "feature_columns": bundle["feature_columns"],
            "base_isolation_features": bundle["isolation_base_features"],
            "contextual_isolation_features": (
                contextual_bundle["feature_columns"]
                if contextual_bundle is not None else []
            ),
            "topology_enabled": topology_enabled,
            "topology_status": topology_reason,
            "topology_residual_features": TOPOLOGY_FEATURES,
            "peer_level": peer_level,
            "cadence_seconds": CADENCE_SECONDS,
            "dispersion_window_seconds": feature_manifest["dispersion_window_seconds"],
            "seasonal_periods": feature_manifest["seasonal_periods"],
            "threshold_method": (
                "empirical daily-block maxima from a late calibration slice "
                "not used for model fitting"
            ),
        }
        write_json(output / "model_manifest.json", model_manifest)

thresholds = pd.read_parquet(OUTPUT_ROOT / "calibration_thresholds.parquet")
display(thresholds)


## 5. Acceptance


In [ ]:
model_manifest = read_json(OUTPUT_ROOT / "model_manifest.json")
assert model_manifest["calibration_only_fit"] is True
assert model_manifest["truth_files_read"] == []
assert {"rapid_residual", "drift_cusum"} <= set(model_manifest["channels"])

display(pd.Series({
    "primary_channels_available": [
        name for name in model_manifest["primary_channels"]
        if name in model_manifest["channels"]
    ],
    "topology_status": model_manifest["topology_status"],
    "challengers_available": [
        name for name in model_manifest["challenger_channels"]
        if name in model_manifest["channels"]
    ],
}, name="result").to_frame())
print("PASS — calibration-frozen scores and thresholds are ready")
print("Next: 07_CHALLENGER_MODELS.ipynb")
